# Mutatierapport Etten-Leur: 2022 → 2025

Stadsbrede scan (5,5 × 5 km, ~27.800 BAG-panden) op basis van de winterluchtfoto's
`2022_orthoHR` en `2025_orthoHR`.

**Methode**
1. **BAG-mutaties** — nieuwbouw (bouwjaar ≥ 2022), in aanbouw, sloopvergunningen, verbouwingen.
2. **Visuele verschilscore per pand** — beide foto's genormaliseerd én *verschuivings-tolerant*
   vergeleken: per pixel telt de kleinste afwijking over verschuivingen tot ±8 px. Dat dempt
   de *omvalling* (daken "leunen" per jaargang een andere kant op doordat het vliegtuig
   ergens anders vloog) die anders valse verschillen geeft.
3. **Kruising** — visuele detecties worden gesplitst in *verklaard door de BAG* en
   *onverklaard*: dat laatste is de werkvoorraad voor een taxateur.

Vereiste downloads (herstartbaar, samen ± 1 GB):
```
python scripts/02_download_tiles.py --bbox 101500,395000,107000,400000 --laag 2022_orthoHR --map-naam el_2022
python scripts/02_download_tiles.py --bbox 101500,395000,107000,400000 --laag 2025_orthoHR --map-naam el_2025
```

In [ ]:
import io, json, sys
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw, ImageFilter
from shapely.geometry import shape

REPO = Path.cwd().resolve()
if REPO.name == 'notebooks':
    REPO = REPO.parent
sys.path.insert(0, str(REPO / 'scripts'))
from common import laad_config
from pdok import fetch_bag_panden, make_session, wms_get_map

cfg = laad_config(REPO / 'config.yaml')
DATA = REPO / cfg['paden']['data']
RES = cfg['luchtfoto']['resolutie']
PX = cfg['luchtfoto']['tegelgrootte']
STAP = PX * RES

GEBIED = [101500, 395000, 107000, 400000]     # heel Etten-Leur
OUD_LAAG, NIEUW_LAAG = '2022_orthoHR', '2025_orthoHR'
OUD_MAP, NIEUW_MAP = DATA / 'el_2022', DATA / 'el_2025'
BAG_ALLE = DATA / 'bag' / 'el_panden_alle.geojson'
DREMPEL_PCT = 99          # percentiel van de verschilscore dat als 'detectie' telt
MARKEER = '#FFD400'
BLAUW, GRIJS, AMBER = '#2563EB', '#94A3B8', '#F59E0B'
sessie = make_session()

for m in (OUD_MAP, NIEUW_MAP):
    assert (m / 'tiles.json').exists(), f'{m} ontbreekt — zie de downloadregels hierboven'
idx_oud = json.loads((OUD_MAP / 'tiles.json').read_text())
idx_nieuw = json.loads((NIEUW_MAP / 'tiles.json').read_text())

if not BAG_ALLE.exists():
    feats = fetch_bag_panden(sessie, cfg['bag']['wfs_url'], tuple(GEBIED), alleen_in_gebruik=False)
    BAG_ALLE.write_text(json.dumps({'type': 'FeatureCollection', 'bbox_rd': GEBIED, 'features': feats}))
panden = json.loads(BAG_ALLE.read_text())['features']
print(f'{len(panden)} BAG-panden, {len(set(idx_oud) & set(idx_nieuw))} tegelparen geladen')

## 1. Wat de BAG registreert (2022–2025)

In [ ]:
nieuwbouw = sorted((f for f in panden if 2022 <= (f['properties'].get('bouwjaar') or 0) <= 2025),
                   key=lambda f: f['properties']['bouwjaar'])
in_aanbouw = [f for f in panden if (f['properties'].get('bouwjaar') or 0) >= 2026
              or f['properties'].get('status') == 'Bouw gestart']
sloop = [f for f in panden if 'loop' in (f['properties'].get('status') or '')]
verbouwing = [f for f in panden if f['properties'].get('status') == 'Verbouwing pand']
vergund = [f for f in panden if f['properties'].get('status') == 'Bouwvergunning verleend']

per_jaar = Counter(f['properties']['bouwjaar'] for f in nieuwbouw)
jaren = sorted(per_jaar)
fig, ax = plt.subplots(figsize=(7, 3.2))
ax.bar([str(j) for j in jaren], [per_jaar[j] for j in jaren], color=BLAUW, width=0.55)
for i, j in enumerate(jaren):
    ax.text(i, per_jaar[j] + max(per_jaar.values()) * 0.02, str(per_jaar[j]),
            ha='center', color='#555555', fontsize=9)
ax.set_title('Nieuwbouw in Etten-Leur per bouwjaar (BAG)', loc='left', fontsize=11)
ax.set_ylabel('panden')
ax.spines[['top', 'right']].set_visible(False)
ax.tick_params(length=0)
plt.tight_layout(); plt.show()

print(f'Nieuw gebouwd 2022-2025:  {len(nieuwbouw)} panden')
print(f'In aanbouw (2025-foto):   {len(in_aanbouw)} panden')
print(f'Sloopvergunning verleend: {len(sloop)} panden')
print(f'Verbouwing pand:          {len(verbouwing)} panden')
print(f'Bouwvergunning verleend:  {len(vergund)} panden')

## 2. Visuele verschilscore per pand (verschuivings-tolerant)

Per tegelpaar: licht blurren, per tegel normaliseren (z-score, dempt lichtverschil), en per
pixel de kleinste afwijking nemen over 9 verschuivingen van ±8 px — zo telt een dak dat door
omvalling een halve meter "verschoven" lijkt niet als verandering. De score per pand is het
gemiddelde binnen de pandcontour-box. Reken op enkele minuten voor de hele stad.

In [ ]:
GRID_X, GRID_Y = GEBIED[0], GEBIED[1]
def tegel_id_voor(x, y):
    return f't_{int((x - GRID_X) // STAP):04d}_{int((y - GRID_Y) // STAP):04d}'

def laad_genorm(pad):
    beeld = Image.open(pad).convert('L').filter(ImageFilter.GaussianBlur(1.5))
    a = np.asarray(beeld, dtype=np.float32)
    return (a - a.mean()) / (a.std() + 1e-6)

per_tegel = defaultdict(list)
for f in panden:
    geom = shape(f['geometry'])
    if geom.area < 25:
        continue
    per_tegel[tegel_id_voor(geom.centroid.x, geom.centroid.y)].append((f, geom))

paren = [tid for tid in per_tegel if tid in idx_oud and tid in idx_nieuw]
VERSCHUIVINGEN = [(dx, dy) for dx in (-8, 0, 8) for dy in (-8, 0, 8)]
scores = []
from tqdm.auto import tqdm
for tid in tqdm(paren, desc='tegelparen scoren'):
    a = laad_genorm(OUD_MAP / idx_oud[tid]['image'])
    b = laad_genorm(NIEUW_MAP / idx_nieuw[tid]['image'])
    minverschil = np.full_like(a, np.inf)
    for dx, dy in VERSCHUIVINGEN:
        d = np.abs(np.roll(b, (dy, dx), axis=(0, 1)) - a)
        np.minimum(minverschil, d, out=minverschil)
    bbox = idx_oud[tid]['bbox']
    for f, geom in per_tegel[tid]:
        gx0, gy0, gx1, gy1 = geom.bounds
        x0 = max(0, int((gx0 - bbox[0]) / RES)); x1 = min(PX, int((gx1 - bbox[0]) / RES))
        y0 = max(0, int((bbox[3] - gy1) / RES)); y1 = min(PX, int((bbox[3] - gy0) / RES))
        if x1 - x0 < 12 or y1 - y0 < 12:
            continue
        scores.append((float(minverschil[y0:y1, x0:x1].mean()), f, geom))

print(f'{len(scores)} panden gescoord over {len(paren)} tegelparen')

## 3. Resultaat: gedetecteerd, verklaard door BAG, onverklaard

In [ ]:
waarden = np.array([s for s, _, _ in scores])
drempel = float(np.percentile(waarden, DREMPEL_PCT))

bag_verklaard_ids = set()
for groep in (nieuwbouw, in_aanbouw, sloop, verbouwing, vergund):
    bag_verklaard_ids |= {f['properties']['identificatie'] for f in groep}

detecties = [(s, f, g) for s, f, g in scores if s >= drempel]
verklaard = [d for d in detecties if d[1]['properties']['identificatie'] in bag_verklaard_ids]
onverklaard = [d for d in detecties if d[1]['properties']['identificatie'] not in bag_verklaard_ids]
onverklaard.sort(key=lambda d: -d[0])

bewaar_map = DATA / 'mutaties_preview'
bewaar_map.mkdir(exist_ok=True)

# Verdeling van de scores met de detectiedrempel
fig, ax = plt.subplots(figsize=(7.5, 3))
ax.hist(waarden, bins=80, color=BLAUW)
ax.axvline(drempel, color='#555555', linestyle='--', linewidth=1.2)
ax.text(drempel, ax.get_ylim()[1] * 0.88, f'  drempel (P{DREMPEL_PCT}) = {drempel:.2f}',
        color='#555555', fontsize=9)
ax.set_title('Verdeling van de verschilscores (alle panden)', loc='left', fontsize=11)
ax.set_xlabel('verschilscore'); ax.set_ylabel('panden')
ax.spines[['top', 'right']].set_visible(False)
ax.tick_params(length=0)
plt.tight_layout(); plt.show()

# Trechter: van scan naar werkvoorraad
stappen = [('Panden geanalyseerd', len(scores), GRIJS),
           (f'Visueel gedetecteerd (score ≥ P{DREMPEL_PCT})', len(detecties), BLAUW),
           ('Verklaard door BAG-registratie', len(verklaard), BLAUW),
           ('Onverklaard → werkvoorraad taxateur', len(onverklaard), AMBER)]
fig, ax = plt.subplots(figsize=(7.5, 2.8))
posities = range(len(stappen))
ax.barh(posities, [n for _, n, _ in stappen], height=0.55, color=[k for _, _, k in stappen])
ax.set_yticks(posities, [naam for naam, _, _ in stappen])
ax.invert_yaxis()
ax.set_xscale('log')
for i, (_, n, _) in enumerate(stappen):
    ax.text(n * 1.12, i, str(n), va='center', color='#555555', fontsize=9)
ax.set_title('Van scan naar werkvoorraad (logaritmische as)', loc='left', fontsize=11)
ax.spines[['top', 'right']].set_visible(False)
ax.tick_params(length=0)
plt.tight_layout()
fig.savefig(bewaar_map / 'trechter.jpg', dpi=110, bbox_inches='tight')
plt.show()

## 4. Voor/na-galerijen (gele box = het pand in kwestie)

In [ ]:
def beeldpaar_rond(geom):
    cx, cy = geom.centroid.x, geom.centroid.y
    tid = tegel_id_voor(cx, cy)
    if tid in idx_oud and tid in idx_nieuw:
        return (Image.open(OUD_MAP / idx_oud[tid]['image']).convert('RGB'),
                Image.open(NIEUW_MAP / idx_nieuw[tid]['image']).convert('RGB'),
                idx_oud[tid]['bbox'])
    halve = STAP / 2
    bbox = (cx - halve, cy - halve, cx + halve, cy + halve)
    oud, nieuw = (Image.open(io.BytesIO(wms_get_map(sessie, cfg['luchtfoto']['wms_url'],
                                                    laag, bbox, PX, PX))).convert('RGB')
                  for laag in (OUD_LAAG, NIEUW_LAAG))
    return oud, nieuw, list(bbox)

def markeer_en_crop(beeld, geom, bbox, marge_m=14):
    gx0, gy0, gx1, gy1 = geom.bounds
    x0, x1 = (gx0 - bbox[0]) / RES, (gx1 - bbox[0]) / RES
    y0, y1 = (bbox[3] - gy1) / RES, (bbox[3] - gy0) / RES
    kopie = beeld.copy()
    ImageDraw.Draw(kopie).rectangle([x0, y0, x1, y1], outline=MARKEER, width=4)
    m = marge_m / RES
    return kopie.crop((max(0, x0 - m), max(0, y0 - m), min(PX, x1 + m), min(PX, y1 + m)))

def toon_voor_na(rijen, titel, bewaar=None):
    if not rijen:
        print('(geen voorbeelden)'); return
    fig, assen = plt.subplots(len(rijen), 2, figsize=(9, 4.4 * len(rijen)), squeeze=False)
    for (geom, onderschrift), (as_o, as_n) in zip(rijen, assen):
        oud, nieuw, bbox = beeldpaar_rond(geom)
        as_o.imshow(markeer_en_crop(oud, geom, bbox))
        as_n.imshow(markeer_en_crop(nieuw, geom, bbox))
        as_o.set_title(f'2022 — {onderschrift}', fontsize=9, loc='left')
        as_n.set_title('2025', fontsize=9, loc='left')
        as_o.axis('off'); as_n.axis('off')
    fig.suptitle(titel, fontsize=12)
    plt.tight_layout()
    if bewaar:
        fig.savefig(bewaar, dpi=110, bbox_inches='tight')
    plt.show()

print('Helpers klaar.')

In [ ]:
rijen = [(g, f'verschilscore {s:.2f}, bouwjaar {f["properties"].get("bouwjaar")}')
         for s, f, g in onverklaard[:4]]
toon_voor_na(rijen, 'Onverklaarde veranderingen — top 4 (werkvoorraad)',
             bewaar=bewaar_map / 'onverklaard.jpg')
rijen = [(g, f'verschilscore {s:.2f}, bouwjaar {f["properties"].get("bouwjaar")}')
         for s, f, g in onverklaard[4:8]]
toon_voor_na(rijen, 'Onverklaarde veranderingen — 5 t/m 8')

In [ ]:
grootste = sorted(nieuwbouw, key=lambda f: -(f['properties'].get('oppervlakte_max') or 0))[:3]
rijen = [(shape(f['geometry']),
          f'bouwjaar {f["properties"]["bouwjaar"]}, {f["properties"].get("oppervlakte_max")} m²')
         for f in grootste]
toon_voor_na(rijen, 'Nieuwbouw — grootste drie (ter validatie van de methode)',
             bewaar=bewaar_map / 'nieuwbouw.jpg')

## 5. Samenvatting

In [ ]:
dekking = sum(1 for tid in per_tegel if tid in idx_oud and tid in idx_nieuw) / max(1, len(per_tegel))
opp = sum(f['properties'].get('oppervlakte_max') or 0 for f in nieuwbouw)
print(f'Etten-Leur (5,5 x 5 km), luchtfoto 2022 -> 2025:')
print(f'- {len(scores)} panden visueel geanalyseerd ({dekking:.0%} tegeldekking)')
print(f'- BAG: {len(nieuwbouw)} nieuw gebouwd (± {opp} m²), {len(in_aanbouw)} in aanbouw,')
print(f'       {len(sloop)} sloopvergunningen, {len(verbouwing)} verbouwingen, {len(vergund)} bouwvergunningen')
print(f'- {len(detecties)} visuele detecties boven drempel P{DREMPEL_PCT} (score >= {drempel:.2f})')
print(f'  waarvan {len(verklaard)} verklaard door de BAG en {len(onverklaard)} ONVERKLAARD')
print(f'- De onverklaarde lijst is de werkvoorraad: vooral zonnepanelen, dakrenovaties en')
print(f'  niet-geregistreerde bouwwerken. Volgende stap: bevestigen/afwijzen in Label Studio.')

---
**Kanttekening bij de methode**: de verschuivings-tolerante score dempt omvalling maar
elimineert hem niet — hoge gebouwen aan de rand van een vluchtstrook kunnen boven de
drempel uitkomen zonder echte verandering. De structurele oplossing is het getrainde
verandermodel ([docs/mutatiedetectie.md](../docs/mutatiedetectie.md)); de bevestigde en
afgewezen detecties uit dít rapport zijn daar de trainingsdata voor.

*Bevat gegevens van PDOK: Luchtfoto Beeldmateriaal Nederland (CC-BY 4.0) en de BAG.*